# Crop Price Watch — Punjab, Pakistan
**Garlic (Local), Carrot, Peas | 22–24 November 2026**

This is the organized version of the original `MidTerm.ipynb`. Every code cell starts with an explanatory comment. No Google Drive is used: Colab uploads and downloads files directly.

**Important:** Part A is for a fresh scraper run only (it gets today's website data). To reproduce the submitted **22–24 November** analysis, skip Part A and upload the three original CSV snapshots in Part B. Never change a collection timestamp to simulate another day.


## Part A — Live web scraping (optional for reproducing the scraping method)

### 1. Import libraries and configure the project

In [ ]:
# Load the required Python libraries and define the crops and region.
# Run this cell first. Install requests/bs4 in Colab only if missing.
import re
import time
import logging
from pathlib import Path
from datetime import datetime, timezone
from email.utils import parsedate_to_datetime
from urllib.parse import urljoin, urlparse
from urllib.robotparser import RobotFileParser
from zoneinfo import ZoneInfo

import pandas as pd
import requests
from bs4 import BeautifulSoup
from google.colab import files

REGION = "Punjab, Pakistan"
CROP_URLS_1 = {
    "Garlic": "https://mandioye.com/amis/commodity/garlic-local",
    "Carrot": "https://mandioye.com/amis/commodity/carrot",
    "Peas": "https://mandioye.com/amis/commodity/peas",
}
CROP_URLS_2 = {
    "Garlic": "https://pakdatahub.com/mandi/garlic_local",
    "Carrot": "https://pakdatahub.com/mandi/carrot",
    "Peas": "https://pakdatahub.com/mandi/peas",
}
TARGET_DATES = ["2026-11-22", "2026-11-23", "2026-11-24"]
RAW_COLUMNS = ["crop", "market", "date", "price", "unit", "price_per_kg",
               "source_url", "scraped_at", "source"]
Path("snapshots").mkdir(exist_ok=True)
logging.basicConfig(level=logging.INFO, format="%(levelname)s: %(message)s")
print("Configured region:", REGION)


### 2. Helper functions: price conversion, published dates, and polite requests

In [ ]:
# Create safe helpers for numbers/dates and a polite HTTP client with robots.txt, delays, and retries.
USER_AGENT = "CropPriceWatchAcademic/1.0 (non-commercial university research)"
REQUEST_DELAY_SECONDS = 2.0
TIMEOUT_SECONDS = (10, 25)
MAX_ATTEMPTS = 3
MAX_MANDIOYE_PAGES = 20
logger = logging.getLogger("crop_price_watch")

def parse_price(value: str) -> float | None:
    """Extract numeric prices from e.g. '33,500', 'Rs 210', or '62.5'."""
    match = re.search(r"\d[\d,]*(?:\.\d+)?", str(value))
    if not match:
        return None
    amount = float(match.group().replace(",", ""))
    return amount if amount > 0 else None


def parse_price_date(value: str) -> str | None:
    """Convert a published price date to ISO yyyy-mm-dd; never guess a date."""
    cleaned = " ".join(str(value).strip().replace(",", " ").split())
    for fmt in ("%d %b %Y", "%d %B %Y", "%Y-%m-%d", "%d-%m-%Y"):
        try:
            return datetime.strptime(cleaned, fmt).date().isoformat()
        except ValueError:
            pass
    return None


def collection_timestamp() -> str:
    """The exact time the request/row was collected (Pakistan local time)."""
    return datetime.now(ZoneInfo("Asia/Karachi")).isoformat(timespec="seconds")


class PoliteFetcher:
    """HTTP client: robots.txt checks, polite pacing, retry and error handling."""

    def __init__(self) -> None:
        self.session = requests.Session()
        self.session.headers.update({
            "User-Agent": USER_AGENT,
            "Accept": "text/html,application/xhtml+xml,text/plain;q=0.9,*/*;q=0.8",
        })
        self.robots: dict[str, RobotFileParser | None] = {}
        self.last_request_at: float | None = None

    def _wait(self) -> None:
        if self.last_request_at is not None:
            remaining = REQUEST_DELAY_SECONDS - (time.monotonic() - self.last_request_at)
            if remaining > 0:
                time.sleep(remaining)
        self.last_request_at = time.monotonic()

    def _request(self, url: str) -> requests.Response | None:
        for attempt in range(1, MAX_ATTEMPTS + 1):
            self._wait()
            try:
                response = self.session.get(url, timeout=TIMEOUT_SECONDS)
                response.raise_for_status()
                return response
            except requests.RequestException as exc:
                logger.warning("Request %d/%d failed: %s (%s)",
                               attempt, MAX_ATTEMPTS, url, exc)
                if attempt < MAX_ATTEMPTS:
                    # Honor numeric Retry-After for 429/503; otherwise back off.
                    retry_after = None
                    resp = getattr(exc, "response", None)
                    if resp is not None:
                        retry_after = resp.headers.get("Retry-After")
                    pause = float(min(30, 2 ** attempt))
                    if retry_after:
                        try:
                            pause = min(60, max(pause, float(retry_after)))
                        except ValueError:
                            try:
                                parsed = parsedate_to_datetime(retry_after)
                                if parsed.tzinfo is None:
                                    parsed = parsed.replace(tzinfo=timezone.utc)
                                seconds = (parsed - datetime.now(timezone.utc)).total_seconds()
                                pause = min(60, max(pause, seconds))
                            except (ValueError, TypeError, OverflowError):
                                pass
                    time.sleep(pause)
        logger.error("Skipping inaccessible URL after retries: %s", url)
        return None

    def _get_robots(self, url: str) -> RobotFileParser | None:
        parsed = urlparse(url)
        base = f"{parsed.scheme}://{parsed.netloc}"
        if base in self.robots:
            return self.robots[base]
        robots_url = base + "/robots.txt"
        logger.info("Checking robots.txt: %s", robots_url)
        # A 404 robots file is conventionally 'no published robots restrictions'.
        # Other HTTP errors or network problems are treated conservatively.
        self._wait()
        try:
            response = self.session.get(robots_url, timeout=TIMEOUT_SECONDS)
            if response.status_code in (404, 410):
                parser = RobotFileParser()
                parser.parse([])
                # Explicit wildcard allow if the site has no robots.txt.
                parser.parse(["User-agent: *", "Allow: /"])
                self.robots[base] = parser
                return parser
            response.raise_for_status()
            parser = RobotFileParser()
            parser.set_url(robots_url)
            parser.parse(response.text.splitlines())
            self.robots[base] = parser
            return parser
        except requests.RequestException as exc:
            logger.error("Cannot verify robots.txt for %s: %s. Skipping site.", base, exc)
            self.robots[base] = None
            return None

    def get_html(self, url: str) -> str | None:
        parser = self._get_robots(url)
        if parser is None:
            return None
        if not parser.can_fetch(USER_AGENT, url):
            logger.warning("robots.txt disallows scraping: %s", url)
            return None
        response = self._request(url)
        return response.text if response is not None else None




### 3. Scrape MandiOye with pagination

In [ ]:
# Find the published price date, follow Next links, and convert FQP from PKR/100kg to PKR/kg.
def get_mandioye_date(soup: BeautifulSoup) -> str | None:
    """Read 'As of DD Mon YYYY' in the page, not the scraping date."""
    text = soup.get_text(" ", strip=True)
    match = re.search(r"\bAs\s+of\s+(\d{1,2}\s+[A-Za-z]+\s+\d{4})\b", text, re.I)
    return parse_price_date(match.group(1)) if match else None


def find_table(soup: BeautifulSoup, required_headers: tuple[str, ...]):
    """Locate the relevant data table by column names, not CSS styling."""
    for table in soup.find_all("table"):
        header_row = table.find("tr")
        if not header_row:
            continue
        headings = " ".join(c.get_text(" ", strip=True).lower()
                            for c in header_row.find_all(["th", "td"]))
        if all(header.lower() in headings for header in required_headers):
            return table
    return None


def next_page_url(soup: BeautifulSoup, current_url: str) -> str | None:
    """Follow the visible 'Next' pagination link on the same commodity path."""
    current = urlparse(current_url)
    for anchor in soup.find_all("a", href=True):
        label = anchor.get_text(" ", strip=True).lower()
        if re.search(r"\bnext\b", label) or anchor.get("rel") == ["next"]:
            candidate = urljoin(current_url, anchor["href"])
            target = urlparse(candidate)
            if (target.netloc == current.netloc and target.path == current.path
                    and target.scheme == "https"):
                return candidate
    return None


def scrape_mandioye(fetcher: PoliteFetcher, crop: str, start_url: str) -> list[dict]:
    """Fetch all pages. FQP is originally quoted in PKR per 100 kilograms."""
    collected: list[dict] = []
    visited: set[str] = set()
    url: str | None = start_url
    page = 0

    while url and url not in visited and page < MAX_MANDIOYE_PAGES:
        visited.add(url)
        page += 1
        html = fetcher.get_html(url)
        if html is None:
            break
        soup = BeautifulSoup(html, "html.parser")
        published_date = get_mandioye_date(soup)
        if not published_date:
            logger.warning("No published 'As of' date found for %s; page ignored", url)
            break

        table = find_table(soup, ("fqp",))
        if table is None:
            logger.warning("Could not find City/FQP table on %s", url)
            break

        page_rows = 0
        for tr in table.find_all("tr"):
            cells = tr.find_all("td")
            if len(cells) < 6:
                continue
            market = cells[1].get_text(" ", strip=True)
            original_price = parse_price(cells[5].get_text(" ", strip=True))
            if not market or original_price is None:
                continue
            collected.append({
                "crop": crop,
                "market": market,
                "date": published_date,
                "price": original_price,
                "unit": "PKR/100kg",
                "source": "mandioye.com",
                "source_url": url,
                "scraped_at": collection_timestamp(),
                "price_per_kg": round(original_price / 100.0, 4),
            })
            page_rows += 1
        logger.info("MandiOye | %s | page %d | price date %s | %d records",
                    crop, page, published_date, page_rows)
        url = next_page_url(soup, url)

    if url and page >= MAX_MANDIOYE_PAGES:
        logger.warning("MandiOye %s: hit pagination safety limit %d", crop, MAX_MANDIOYE_PAGES)
    return collected




### 4. Scrape PakDataHub for Punjab only

In [ ]:
# Parse the second website and keep only Punjab records, preserving published price dates.
def scrape_pakdatahub(fetcher: PoliteFetcher, crop: str, url: str) -> list[dict]:
    """Read public price table; keep only Punjab; price already PKR/kg."""
    html = fetcher.get_html(url)
    if html is None:
        return []
    soup = BeautifulSoup(html, "html.parser")
    table = find_table(soup, ("market", "province", "date"))
    if table is None:
        logger.warning("Could not find Market/Province/Date table: %s", url)
        return []

    collected: list[dict] = []
    for tr in table.find_all("tr"):
        cells = tr.find_all("td")
        if len(cells) < 5:
            continue
        market = cells[0].get_text(" ", strip=True)
        province = cells[1].get_text(" ", strip=True)
        if province.casefold() != "punjab":
            continue
        original_price = parse_price(cells[2].get_text(" ", strip=True))
        published_date = parse_price_date(cells[4].get_text(" ", strip=True))
        if not market or original_price is None or not published_date:
            continue
        collected.append({
            "crop": crop,
            "market": market,
            "date": published_date,
            "price": original_price,
            "unit": "PKR/kg",
            "source": "pakdatahub.com",
            "source_url": url,
            "scraped_at": collection_timestamp(),
            "price_per_kg": round(original_price, 4),
        })
    logger.info("PakDataHub | %s | Punjab | %d records", crop, len(collected))
    return collected




### 5. Collect both sources and download a timestamped raw snapshot

In [ ]:
# Run the web scraper now, save a uniquely timestamped raw CSV, and download it to your computer.
# Run this ONCE per actual collection day. Do NOT rerun for fake historical days.
fetcher = PoliteFetcher()
raw_rows = []
for crop, url in CROP_URLS_1.items():
    raw_rows.extend(scrape_mandioye(fetcher, crop, url))
for crop, url in CROP_URLS_2.items():
    raw_rows.extend(scrape_pakdatahub(fetcher, crop, url))

if not raw_rows:
    raise RuntimeError("No rows collected: inspect robots.txt, connectivity and the page structure.")

run_time = datetime.now(ZoneInfo("Asia/Karachi")).strftime("%Y-%m-%d_%H-%M-%S")
snapshot_path = Path("snapshots") / f"full_snapshot_{run_time}.csv"
raw_df = pd.DataFrame(raw_rows).reindex(columns=RAW_COLUMNS)
# Mode 'x' will fail safely rather than replacing an existing snapshot.
with snapshot_path.open("x", newline="", encoding="utf-8-sig") as handle:
    raw_df.to_csv(handle, index=False)
print("Downloaded raw snapshot:", snapshot_path.name, "| Rows:", len(raw_df))
print(raw_df.groupby("source").size())
files.download(str(snapshot_path))


## Part B — Reproduce and validate the submitted 22–24 November data

### 6. Upload the three original timestamped snapshots

In [ ]:
# Upload the three real snapshot files from your computer; no Google Drive connection is needed.
# The original files can be named "snapshot 1 .csv", "snapshot 2.csv", "snapshot 3 .csv".
from google.colab import files
uploaded = files.upload()
snapshot_data = {}
for filename in uploaded:
    frame = pd.read_csv(filename, encoding="utf-8-sig")
    required = {"crop", "market", "date", "price_per_kg", "source", "source_url", "scraped_at"}
    if not required.issubset(frame.columns):
        print("Skipped non-snapshot file:", filename)
        continue
    collection_days = set(frame["scraped_at"].astype(str).str[:10])
    if len(collection_days) != 1:
        raise ValueError(f"Multiple collection days in {filename}: {collection_days}")
    collection_day = next(iter(collection_days))
    if collection_day in snapshot_data:
        raise ValueError(f"Two snapshots were uploaded for the same collection day: {collection_day}")
    snapshot_data[collection_day] = frame.copy()
    print(filename, "->", collection_day, "|", len(frame), "records")

if set(snapshot_data) != set(TARGET_DATES):
    raise ValueError(f"Expected snapshots for {TARGET_DATES}, found {sorted(snapshot_data)}")
print("All three dated snapshots loaded successfully.")


### 7. Combine raw snapshots and inspect coverage

In [ ]:
# Combine the three uploaded runs and count their source and actual published price dates.
all_snapshots = pd.concat(
    [snapshot_data[day].assign(snapshot_day=day) for day in TARGET_DATES],
    ignore_index=True
)
print("Raw observations:", len(all_snapshots))
print("\nBy collection day and source:")
print(pd.crosstab(all_snapshots["snapshot_day"], all_snapshots["source"]))
print("\nBy published PRICE date and source (different from collection date):")
print(pd.crosstab(all_snapshots["date"], all_snapshots["source"]))


### 8. Clean names, prices, dates, and duplicate observations

In [ ]:
# Build clean_data.csv using consistent PKR/kg prices and a normalized market-name key.
# Prefer MandiOye when the same crop, market and published price date occur in both sources.
work = all_snapshots.copy()
work["crop"] = work["crop"].astype("string").str.strip().str.title()
work["market"] = (
    work["market"].astype("string")
    .str.replace(r"\s+major$", "", regex=True, case=False)
    .str.strip().str.title()
)
work["date"] = pd.to_datetime(work["date"], errors="coerce").dt.strftime("%Y-%m-%d")
work["price"] = pd.to_numeric(work["price_per_kg"], errors="coerce")
work["source"] = work["source"].astype("string").str.strip()
work = work[work["date"].isin(TARGET_DATES)].copy()
work = work.dropna(subset=["crop", "market", "date", "price", "source", "source_url", "scraped_at"])
work = work[(work["price"] > 0) & (work["market"].str.len() > 0)].copy()
work["market_key"] = work["market"].str.lower().str.replace(r"[^a-z0-9]", "", regex=True)
work["priority"] = (work["source"] == "mandioye.com").astype(int)
work = work.sort_values("priority", ascending=False, kind="stable")

# Inspect conflicts before deduplicating; do not hide different prices silently.
conflicts = work.groupby(["crop", "market_key", "date"])["price"].nunique()
print("Groups with conflicting published prices:", int((conflicts > 1).sum()))
print("If greater than zero, review those groups and document source preference.")

clean = work.drop_duplicates(["crop", "market_key", "date"], keep="first").copy()
clean["unit"] = "PKR/kg"
clean = clean[["crop", "market", "date", "price", "unit", "source", "source_url", "scraped_at"]]
clean = clean.sort_values(["date", "crop", "market"]).reset_index(drop=True)
clean.to_csv("clean_data.csv", index=False, encoding="utf-8-sig")
print("Clean rows:", len(clean), "| Removed:", len(work) - len(clean))
print(clean.groupby("date").size())


### 9. Validate the cleaned CSV

In [ ]:
# Test missing values, valid prices, units, and duplicates including market spelling variations.
check = clean.copy()
check["market_key"] = check["market"].str.lower().str.replace(r"[^a-z0-9]", "", regex=True)
print("Missing by field:\n", check.isna().sum())
print("Invalid prices:", int((check["price"] <= 0).sum()))
print("Duplicate crop/market/date:", int(check.duplicated(["crop", "market_key", "date"]).sum()))
print("Units:\n", check["unit"].value_counts())
print("Coverage:\n", pd.crosstab(check["date"], check["source"]))
assert not check.isna().any().any(), "Unexpected missing values"
assert (check["price"] > 0).all(), "Invalid price"
assert not check.duplicated(["crop", "market_key", "date"]).any(), "Duplicate price record"
assert check["unit"].eq("PKR/kg").all(), "Unit mismatch"


### 10. Q1 — Find the most expensive crop on 24 November

In [ ]:
# Answer Q1 using mean prices across all available markets on 24 November 2026.
q1 = (clean[clean["date"] == "2026-11-24"]
      .groupby("crop")["price"].agg(["count", "mean"])
      .sort_values("mean", ascending=False))
print(q1.round(2))
print("Most expensive crop:", q1.index[0])


### 11. Q2 — Compare movement at the SAME markets across all three days

In [ ]:
# Calculate average cumulative absolute daily percentage movement for matched markets.
# Use MandiOye only, because PakDataHub's published dates are not current for both later days.
mandi = clean[clean["source"] == "mandioye.com"].copy()
price_wide = mandi.pivot_table(index=["crop", "market"], columns="date", values="price", aggfunc="first")
price_wide = price_wide.reindex(columns=TARGET_DATES).dropna().copy()
a, b, c = TARGET_DATES
price_wide["absolute_movement_pkr"] = abs(price_wide[b] - price_wide[a]) + abs(price_wide[c] - price_wide[b])
price_wide["movement_pct"] = (abs(price_wide[b]-price_wide[a])/price_wide[a]
                              + abs(price_wide[c]-price_wide[b])/price_wide[b]) * 100
price_wide["net_change_pkr"] = price_wide[c] - price_wide[a]
q2_summary = (price_wide.reset_index().groupby("crop")
              .agg(matched_markets=("market", "nunique"),
                   avg_absolute_movement_pkr=("absolute_movement_pkr", "mean"),
                   avg_movement_pct=("movement_pct", "mean"),
                   avg_net_change_pkr=("net_change_pkr", "mean"))
              .sort_values("avg_movement_pct", ascending=False))
print(q2_summary.round(2))
print("Largest relative movement:", q2_summary.index[0])
q2_tableau = (price_wide[TARGET_DATES].reset_index()
              .melt(id_vars=["crop", "market"], value_vars=TARGET_DATES,
                    var_name="date", value_name="price"))
q2_tableau.to_csv("q2_matched_markets.csv", index=False, encoding="utf-8-sig")
print("Q2 Tableau rows:", len(q2_tableau))


### 12. Q3 — Compare markets for the same crop on 24 November

In [ ]:
# Summarize how the price of each crop varies across markets on 24 November.
q3_day = clean[clean["date"] == "2026-11-24"].copy()
q3 = (q3_day.groupby("crop")["price"]
      .agg(markets="count", min_price="min", mean_price="mean", max_price="max"))
print(q3.round(2))
print("Note: PakDataHub has no same-day 23/24 price observations in the supplied snapshots.")
print("A full same-day source-vs-source comparison is not supported for the last two days.")


### 13. Export Tableau files and download them to your computer

In [ ]:
# Save the requested clean data and two consistent CSV files for Tableau, then download them.
rename_cols = {"crop":"Crop", "market":"Market", "date":"Date", "price":"Price",
               "unit":"Unit", "source":"Source", "source_url":"Source URL", "scraped_at":"Scraped At"}
clean.rename(columns=rename_cols).to_csv("dashboard_data.csv", index=False, encoding="utf-8-sig")
for filename in ["clean_data.csv", "dashboard_data.csv", "q2_matched_markets.csv"]:
    print("Downloading:", filename)
    files.download(filename)


## How this matches the Tableau workbook

- **Q1 Bar Chart:** `dashboard_data.csv`, AVG(Price), filtered to 24 Nov.
- **Q2 Line Chart:** `q2_matched_markets.csv`, 111 crop–market pairs × 3 dates = 333 observations.
- **Q3 Heatmap:** `dashboard_data.csv`, 24 Nov, colored by a Tableau calculated field comparing the market price with the average price of the same crop on that date.
- **Q3 Box Plot:** `dashboard_data.csv`, 24 Nov, Market on Detail to show price distribution and outliers.
- **Submitted ZIP:** `scraper.py`, `snapshots/`, `clean_data.csv`, `dashboard.twbx`, `NOTES.md`.

**No Google Drive code is needed.** Upload/download is done directly using Colab's `files` tools. The `scraper.py` file is standalone for reproducing the collection on a future date; historical data come only from the original timestamped snapshots.
